# Multi Vector Retriever — 요약으로 검색, 원문으로 반환

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **MultiVectorRetriever**의 "검색용 표현"과 "반환용 문서"를 분리하는 전략 이해하기
- LLM으로 문서 요약을 생성하고 벡터스토어에 저장하는 방법 익히기
- `InMemoryStore`에 원문을 저장하고 연결하는 구조 구현하기

## 사전 지식

- `uuid`로 고유 ID를 생성하는 방법
- ParentDocumentRetriever의 부모-자식 문서 개념

---

## 전략 비교

### 전략 1: 요약 기반 검색
원문이 복잡하고 길 때 명확한 요약으로 검색 정확도를 높여요:

```
요약(검색용): "Word2Vec은 단어를 벡터로 변환하는 기법"
원문(반환용): "Word2Vec은... (전체 2000자 상세 설명)"
```

### 전략 2: 가상 질문 기반 검색
문서에서 예상되는 질문을 미리 생성해 저장하면 실제 사용자 질문과 더 잘 매칭돼요:

```
가상 질문: "Transformer란 무엇인가요?"
원문(반환용): "Transformer는 Attention 메커니즘을 사용..."
```

> **실무 팁**: 문서 수가 적을 때(5~10개)는 LLM 요약 비용이 크지 않지만, 수천 개 문서라면 비용을 사전에 계산해보세요.

In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [4]:
from ipaddress import summarize_address_range
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ---------------------------------------------------
# 문서 요약을 생성하고 MultiVectorRetriever 구조 설정
# ---------------------------------------------------

# ============================================================
# TODO: ChatOpenAI로 각 문서의 요약을 생성하세요 (처음 5개 청크만)
# 힌트: chain.invoke({"doc": d.page_content}) — 한 문장 요약
# 예상 결과: 5개 요약 생성 완료 메시지 출력
# ============================================================

from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

# 문서 로드 및 분할 (처음 5개만 사용 — LLM 요약 비용 절감)
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
docs = text_splitter.split_documents(documents)[:5]

# LLM 및 요약 체인
# temperature=0: 요약이 일관되게 생성되도록 설정
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")
chain = (
    ChatPromptTemplate.from_template("다음 문서를 한 문장으로 요약하세요: \n\n{doc}")
    | llm
    | StrOutputParser()
)
# 요약 생성 (검색용)
summaries = [chain.invoke({"doc": d.page_content}) for d in docs]
print(f' ==> [Line 41]: \033[38;2;97;162;9m[summaries]\033[0m({type(summaries).__name__}) = \033[38;2;196;242;243m{summaries}\033[0m')



 ==> [Line 41]: [summaries](list) = ['Scikit-learn은 Python을 위한 기계 학습 라이브러리로, 다양한 알고리즘과 도구를 제공하여 연구자와 개발자가 데이터 과학 문제를 효율적으로 해결할 수 있도록 돕는 오픈 소스 프로젝트입니다.', 'Scikit-learn은 기계 학습 초보자에게 유용한 학습 자원을 제공하며, NLP는 인간 언어를 이해하고 해석하는 기술로 다양한 분야에서 활용되지만, 자연어의 모호성과 다양성을 처리하는 것이 주요 과제입니다.', '최근 딥 러닝 기반의 NLP 모델, 특히 변환기 아키텍처의 발전으로 자연어 이해와 생성이 크게 향상되어 사람과 컴퓨터 간의 소통 방식이 변화하고 있으며, SciPy는 과학 계산을 위한 파이썬 라이브러리로 수학 및 과학 분야에서 널리 사용되고 있습니다.', 'SciPy는 최적화, 선형 대수, 적분 등 다양한 수학적 모듈을 제공하여 연구자와 엔지니어가 복잡한 과학적 문제를 효율적으로 해결할 수 있도록 돕는 오픈 소스 라이브러리로, Python의 다른 도구들과 결합하여 강력한 데이터 분석 및 시각화 기능을 지원합니다.', "Hugging Face는 자연어 처리(NLP) 분야에서 오픈 소스 도구와 라이브러리, 특히 'Transformers'를 제공하여 최신 AI 기술을 쉽게 접근하고 활용할 수 있도록 지원하는 기술 회사입니다."]


In [7]:
# ---------------------------------------------------
# MultiVectorRetriever 생성 — 요약을 검색용, 원문을 반환용으로 분리 저장
# ---------------------------------------------------

# ============================================================
# TODO: MultiVectorRetriever를 생성하고 요약(vectorstore)과 원문(docstore)을 연결하세요
# 힌트: uuid로 문서 ID를 생성하고 metadata에 id_key로 연결
# 예상 결과: vectorstore에 요약 저장, docstore에 원문 저장 완료 메시지 출력
# ============================================================

# MultiVectorRetriever 생성
# vectorstore: 요약 임베딩 저장 (검색용)
from langchain.retrievers import multi_vector


vectorstore = FAISS.from_texts(["init"], OpenAIEmbeddings(model="text-embedding-3-small"))

# InMemoryStore: 원문 저장 (반환용)
store = InMemoryStore()

# id_key: 요약과 원문을 연결하는 메타데이터 키
id_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key
)


# 문서 ID 생성 — uuid로 각 문서에 고유 ID 부여
doc_ids = [str(uuid.uuid4()) for _ in docs]


# 요약을 vectorstore에 저장 (검색용)
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]}) for i, s in enumerate(summaries)
]

multi_vector_retriever.vectorstore.add_documents(summary_docs)
multi_vector_retriever.docstore.mset(list(zip(doc_ids, docs)))

# 원문을 docstore에 저장 (반환용) — id를 키로 연결


In [8]:
# 검색

# 아래에 코드를 작성하세요
query = "머신러닝 라이브러리에 대해 알려줘."

retrieved_docs = multi_vector_retriever.invoke(query)

print(f":memo: 쿼리: {query}\n")
print("="*80)
print("[MultiVectorRetriever 결과]")
print("="*80)
for i, doc in enumerate(retrieved_docs, 1):
    print(f"[문서 {i}]")
    print(f"길이: {len(doc.page_content)}자 ← 원문 (상세)")
    print(f"내용: {doc.page_content[:150]}...")
    print("-"*80)

:memo: 쿼리: 머신러닝 라이브러리에 대해 알려줘.

[MultiVectorRetriever 결과]
[문서 1]
길이: 738자 ← 원문 (상세)
내용: Scikit Learn

Scikit-learn은 Python 언어를 위한 또 다른 핵심 라이브러리로, 기계 학습의 다양한 알고리즘을 구현하기 위해 설계되었습니다. 이 라이브러리는 2007년 David Cournapeau에 의해 프로젝트가 시작되었으며, 그 후로 커뮤니...
--------------------------------------------------------------------------------
[문서 2]
길이: 780자 ← 원문 (상세)
내용: 최근 몇 년 동안, 딥 러닝 기반의 NLP 모델, 특히 변환기(Transformer) 아키텍처와 같은 신경망 모델의 발전으로 인해, 자연어 이해와 생성 분야에서 놀라운 진보가 이루어졌습니다. 이러한 모델들은 대규모의 언어 데이터에서 학습하여, 문장의 문맥을 효과적으로 ...
--------------------------------------------------------------------------------
[문서 3]
길이: 783자 ← 원문 (상세)
내용: HuggingFace

Hugging Face는 자연어 처리(NLP) 분야에서 선도적인 역할을 하는 기술 회사로, 인공 지능(AI) 연구와 응용 프로그램 개발을 위한 다양한 오픈 소스 도구와 라이브러리를 제공합니다. 2016년에 설립된 이 회사는 특히 'Transfor...
--------------------------------------------------------------------------------
[문서 4]
길이: 670자 ← 원문 (상세)
내용: 기계 학습 분야에 있어서, Scikit-learn은 특히 초보자에게 친화적인 학습 자원을 제공함으로써, 복잡한 이론과 알고리즘을 이해하는 데 중요한 역할을 합니다. 이러한 접근성과 범용성은 Scikit-lear

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 임베딩 대상(요약/가상질문)과 반환 대상(원본)을 분리 |
| 두 가지 전략 | 요약 기반 — 긴 문서에 적합 / 가상 질문 기반 — QA 데이터셋에 적합 |
| 저장 구조 | VectorStore에 요약/질문, InMemoryStore에 원본 문서 |
| 연결 키 | `uuid` — VectorStore의 메타데이터와 InMemoryStore의 키를 연결 |
| 전처리 비용 | 인덱싱 시 LLM 호출 필요 (요약 생성 단계) |

### 전략 선택 가이드

| 전략 | 적합한 상황 | 인덱싱 비용 |
|------|-------------|-------------|
| 요약 기반 (Summarization) | 문서가 길고 핵심만 검색할 때 | 높음 (LLM 요약 필요) |
| 가상 질문 (Hypothetical Questions) | 예상 질문이 명확한 FAQ형 데이터 | 높음 (LLM 생성 필요) |
| 소형 청크 (Small Chunks) | 빠른 검색이 우선일 때 | 낮음 |

### 다음 노트북 예고

**08-SelfQueryRetriever** — 자연어 질문에서 메타데이터 필터(연도, 카테고리, 평점 등)를 LLM이 자동으로 추출해 구조화된 검색을 수행하는 방법을 배워요.